# Investment Research Agent
**AAI-520 Final Project — University of San Diego**

Team: Viviana Garzon · Obinna Edeh · Dina Othman

---

## Project Goal
Build a multi-agent AI system that researches any stock ticker and generates a professional investment report automatically — pulling live price data, financials, and news, then analyzing and refining the output using three workflow patterns:

| Workflow | What it does |
|---|---|
| **Prompt Chaining** | Ingest news >> clean >> classify >> extract facts >>summarize |
| **Routing** | Classify content and dispatch to the right specialist agent |
| **Evaluator-Optimizer** | Score a draft, give feedback, rewrite — repeat until quality passes |


## 0. Setup

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

TICKER = "AAPL"
print("Setup complete. Repo root:", repo_root)

## 1. Data Tools
Verify the data tools work independently before running any agent.

In [ ]:
from tools.yfinance_tool import get_stock_data, get_financials

market = get_stock_data(TICKER)
print(f"Ticker:        {market['ticker']}")
print(f"Current price: ${market['current_price']:,.2f}")
print(f"Change today:  {market['change_pct']:+.2f}%")
print(f"52W range:     ${market['week_52_low']} – ${market['week_52_high']}")
print(f"MA50:          {market['ma_50']}")
print(f"History rows:  {len(market['history'])}")

In [ ]:
fin = get_financials(TICKER)
print(f"Sector:        {fin['sector']}")
print(f"Revenue:       {fin['revenue']}")
print(f"Net income:    {fin['net_income']}")
print(f"P/E ratio:     {fin['pe_ratio']}")

In [ ]:
from tools.news_tool import fetch_news

articles = fetch_news(TICKER, lookback_days=7, max_articles=5)
print(f"Articles found: {len(articles)}\n")
for a in articles:
    print(f"  [{a['source']}] {a['title']}")

## 2. Market Agent
Analyzes price trends, moving averages, and volume with no LLM required.

In [ ]:
from agents.market_agent import MarketAgent

market_agent = MarketAgent()
market_result = market_agent.analyze(TICKER, period="3mo")

print(f"Ticker:          {market_result['ticker']} as of {market_result['as_of']}")
print(f"Price:           ${market_result['current_price']:,.2f}")
print(f"Period return:   {market_result['period_return_pct']:+.2f}%")
print(f"Trend:           {market_result['trend']}")
print(f"Crossover:       {market_result['crossover']}")
print(f"Volume ratio:    {market_result['volume_ratio']}x 20-session avg")
print(f"\nSummary: {market_result['summary']}")

In [ ]:
from IPython.display import SVG, display
from tools.market_visualization import save_price_chart

chart_path = save_price_chart(market_result, repo_root / "market_price_chart.svg")
display(SVG(filename=str(chart_path)))

## 3. Earnings Agent
Pulls financial statement data and computes key metrics including revenue, net income, margins, and valuation ratios.

In [ ]:
from agents.earnings_agent import EarningsAgent

def fmt_currency(v, d=0):
    return f"${v:,.{d}f}" if v is not None else "N/A"

def fmt_number(v, d=2):
    return f"{v:.{d}f}" if v is not None else "N/A"

def fmt_pct(v):
    return f"{v:.2f}%" if v is not None else "N/A"

earnings_agent = EarningsAgent()
earnings_result = earnings_agent.analyze(TICKER)

print(f"Ticker:             {earnings_result['ticker']}")
print(f"Sector:             {earnings_result['sector'] or 'N/A'}")
print(f"Industry:           {earnings_result['industry'] or 'N/A'}")
print(f"Revenue:            {fmt_currency(earnings_result['revenue'])}")
print(f"Net Income:         {fmt_currency(earnings_result['net_income'])}")
print(f"EPS:                {fmt_currency(earnings_result['eps'], 2)}")
print(f"P/E Ratio:          {fmt_number(earnings_result['pe_ratio'])}")
print(f"Forward P/E:        {fmt_number(earnings_result['forward_pe'])}")
print(f"Net Profit Margin:  {fmt_pct(earnings_result['net_margin_pct'])}")
print("\nFinancial Signals:")
for signal in earnings_result['signals']:
    print(f"  - {signal}")

## 4. News Agent
Fetches recent articles, classifies each by financial topic, and summarizes key themes.

In [ ]:
from agents.news_agent import NewsAgent

news_agent = NewsAgent()
news_articles = news_agent.fetch_and_classify(TICKER, lookback_days=7)

print(f"Articles classified: {len(news_articles)}\n")
for article in news_articles[:5]:
    print(f"  [{article['category']:10s}] {article['title']}")

In [ ]:
news_summary = news_agent.summarize(news_articles)
print("News Summary:\n")
print(news_summary)

In [ ]:
# Combined specialist output: earnings signals + news articles feed into the ResearchAgent
specialist_results = {
    "earnings": earnings_result,
    "news": {
        "articles_analyzed": len(news_articles),
        "summary": news_summary,
    },
}

print(f"Earnings data source:    {earnings_result['data_source']}")
print(f"News articles analyzed:  {len(news_articles)}")
print("Earnings signals:")
for signal in earnings_result['signals']:
    print(f"  - {signal}")

## 5. Workflow 1: Prompt Chaining
Runs a 5-step pipeline on news articles through ingest, preprocess, classify, extract facts, and summarize.

In [ ]:
from workflows.prompt_chaining import run_prompt_chain

chain_result = run_prompt_chain(TICKER, lookback_days=7)

print(f"Articles ingested:    {chain_result['articles_ingested']}")
print(f"After deduplication:  {chain_result['articles_after_dedup']}")
print("\nKey facts:")
for fact in chain_result['key_facts']:
    print(f"  - {fact}")

In [ ]:
print("News Summary:\n")
print(chain_result['summary'])

## 6. Workflow 2: Routing
Classifies a piece of content and routes it to the correct specialist agent.

In [ ]:
import os
import anthropic
from workflows.routing import classify_route

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

samples = [
    "Apple reported quarterly EPS of $1.64, beating estimates by $0.08. Revenue grew 5% YoY.",
    "Analyst upgrades AAPL to Buy with $220 price target citing strong services momentum.",
    "AAPL is trading above its 50-day moving average on above-average volume after a 3% breakout.",
]

for content in samples:
    route = classify_route(content, client)
    print(f"Route: {route:10s} | {content[:80]}")

## 7. Workflow 3: Evaluator-Optimizer
Takes a draft analysis, scores it, and refines it in a loop until quality passes the threshold.

In [ ]:
from workflows.evaluator_optimizer import run_evaluator_optimizer

eval_result = run_evaluator_optimizer(
    draft=chain_result['summary'],
    threshold=8,
    max_iterations=3
)

print(f"Iterations: {eval_result['iterations']}\n")
for h in eval_result['history']:
    print(f"  Iteration {h['iteration']}: score {h['score']}/10")
    print(f"  Feedback: {h['feedback'][:120]}...\n")

In [ ]:
print("Final refined draft:\n")
print(eval_result['final_draft'])

## 8. Research Agent: Full Pipeline
Orchestrates all tools and workflows through planning, data fetching, synthesis, and self-reflection.

In [ ]:
from agents.research_agent import ResearchAgent

agent = ResearchAgent()
report = agent.run(TICKER)

print("Research Plan:")
for i, step in enumerate(report['steps'], 1):
    print(f"  {i}. {step}")

In [ ]:
print("Market Snapshot:")
snap = report['market_snapshot']
print(f"  Price:     ${snap['price']:,.2f} ({snap['change_pct']:+.2f}%)")
print(f"  52W range: ${snap['week_52_low']} – ${snap['week_52_high']}")

print("\nFinancials Snapshot:")
fin = report['financials_snapshot']
print(f"  Revenue:    {fin['revenue']}")
print(f"  Net income: {fin['net_income']}")
print(f"  P/E ratio:  {fin['pe_ratio']}")
print(f"  Sector:     {fin['sector']}")

In [ ]:
print("Investment Analysis:\n")
print(report['analysis'])

In [ ]:
reflection = agent.reflect(report)
print("Self-Critique:\n")
print(reflection)

---
## 9. Conclusion

